In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path
from sklearn.metrics import precision_recall_curve, auc, roc_curve, roc_auc_score
import statsmodels.stats.multitest as sm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
from aquarel import load_theme

theme = load_theme("minimal_light")
theme.apply()
theme.apply_transforms()
mpl.rcParams['axes.spines.left'] = True
mpl.rcParams['axes.spines.right'] = False
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.bottom'] = True
mpl.rcParams['axes.edgecolor'] = 'grey'

In [ ]:
FIGURE_DATA = Path("../figure_data")

ES_INPUT_MAP = {'es0': 3.0, 'es1': 2.0, 'es2': 1.0, 'es3': 0.75, 'es4': 0.5, 'es5': 0.25}

master = pd.read_csv(FIGURE_DATA / 'per_pair_results.csv')

In [ ]:
# CONFIGURATION
power_of_interest = 0.75       # corresponds to effect_size='es3'
es_of_interest    = 'es3'
threshold_points  = [0.05, 0.01, 0.001]
threshold_symbols = ['o', 'd', '*']

METHOD_ORDER = ['SimPhyNI', 'FaST-LMM', 'TreeWAS', 'Pagel', 'Scoary', 'Coinfinder', 'SpydrPick', "Fisher's"]
color_map = matplotlib.colormaps.get_cmap('tab10')
method_colors = {m: color_map(i) for i, m in enumerate(METHOD_ORDER)}

In [ ]:
def collect_pr_roc_data(label_of_interest):
    pr_roc_data = {}

    # Filter to the effect size of interest
    df_es = master[master['effect_size'] == es_of_interest].copy()

    # Map 'TreeWAS Terminal' in data to 'TreeWAS' display name
    # METHOD_ORDER uses 'TreeWAS' which corresponds to 'TreeWAS Terminal' in the data
    data_method_map = {m: m for m in METHOD_ORDER}
    data_method_map['TreeWAS'] = 'TreeWAS Terminal'

    for method in METHOD_ORDER:
        data_method_name = data_method_map[method]
        df_method = df_es[df_es['method'] == data_method_name].copy()

        if df_method.empty:
            continue

        pvalues_np = df_method['raw_pvalue'].fillna(1).astype(float).to_numpy()
        directions_np = df_method['direction'].fillna(0).astype(int).to_numpy()
        pred_direction_np = df_method['pred_direction'].fillna(0).astype(int).to_numpy()

        capped_pvalues = np.maximum(pvalues_np, 1e-300)

        # Apply FDR correction
        corrected_p = sm.multipletests(capped_pvalues, alpha=0.01, method='fdr_by')[1]

        binary_labels = (directions_np == label_of_interest).astype(int)
        if np.sum(binary_labels) == 0:
            continue

        # Convert to -log10 scale, flip sign for wrong predicted direction
        neglog_pvals = -np.log10(capped_pvalues)
        neglog_pvals[pred_direction_np != label_of_interest] *= -1

        # Get PR curve
        precision, recall, pr_thresholds = precision_recall_curve(binary_labels, neglog_pvals)
        pr_auc_value = auc(recall, precision)

        # Trim last element so dimensions align
        precision = precision[:-1]
        recall = recall[:-1]

        # ROC curve
        fpr, tpr, roc_thresholds = roc_curve(binary_labels, neglog_pvals)
        roc_auc_value = roc_auc_score(binary_labels, neglog_pvals)

        pr_roc_data[method] = {
            'precision': precision,
            'recall': recall,
            'pr_thresholds': pr_thresholds,
            'pr_auc': pr_auc_value,
            'fpr': fpr,
            'tpr': tpr,
            'roc_thresholds': roc_thresholds,
            'roc_auc': roc_auc_value,
            'raw_pvalues': capped_pvalues,
            'corrected_pvalues': corrected_p,
        }

    return pr_roc_data

In [ ]:
def plot_pr_curves(pr_roc_data, label_text):
    fig, ax = plt.subplots(figsize=(3, 3))
    for method, data in pr_roc_data.items():
        ax.step(data['recall'], data['precision'], where='post',
                label=f"{method}",
                color=method_colors[method], alpha=0.8, linestyle='dotted')
        thresh = [-np.log(i) for i in threshold_points]
        for symbol, p_thresh in zip(threshold_symbols, thresh):
            idx = (np.abs(data['pr_thresholds'] - p_thresh)).argmin()
            rec = data['recall'][idx]
            prec = data['precision'][idx]
            ax.scatter(rec, prec, color=method_colors[method], marker=symbol, s=60, alpha=.8)
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    plt.tight_layout()
    ax.legend(frameon=False).set_visible(False)
    plt.savefig('fig.svg', format='svg')
    plt.show()

In [ ]:
def plot_pr_curves_zoom(pr_roc_data, label_text):
    fig, ax = plt.subplots(figsize=(3, 3))

    # Main PR curve
    for method, data in pr_roc_data.items():
        ax.step(data['recall'], data['precision'], where='post',
                label=f"{method}",
                color=method_colors[method], alpha=0.8, linestyle='dotted')

        thresh = [-np.log(i) for i in threshold_points]
        for symbol, p_thresh in zip(threshold_symbols, thresh):
            idx = (np.abs(data['pr_thresholds'] - p_thresh)).argmin()
            rec = data['recall'][idx]
            prec = data['precision'][idx]
            ax.scatter(rec, prec, color=method_colors[method], marker=symbol, s=60, alpha=0.8)

    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(0.05, 1.05)
    plt.tight_layout()
    ax.legend(frameon=False).set_visible(False)

    # Add zoomed-in inset
    axins = inset_axes(ax, width="50%", height="50%", loc='lower left',
                       bbox_to_anchor=(.325, .2, 1.05, .45), bbox_transform=ax.transAxes)

    for method, data in pr_roc_data.items():
        axins.step(data['recall'], data['precision'], where='post',
                   color=method_colors[method], alpha=0.8, linestyle='dotted')
        thresh = [-np.log(i) for i in threshold_points]
        for symbol, p_thresh in zip(threshold_symbols, thresh):
            idx = (np.abs(data['pr_thresholds'] - p_thresh)).argmin()
            rec = data['recall'][idx]
            prec = data['precision'][idx]
            axins.scatter(rec, prec, color=method_colors[method], marker=symbol, s=40, alpha=0.8)

    axins.set_xlim(0.60, .95)
    axins.set_ylim(0.85, 1.02)
    axins.tick_params(labelleft=False, labelbottom=False, left=False, bottom=False)
    axins.spines['top'].set_visible(True)
    axins.spines['right'].set_visible(True)
    axins.spines['bottom'].set_visible(True)
    axins.spines['left'].set_visible(True)

    mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="0.5")

    plt.savefig('fig.svg', format='svg')
    plt.show()

In [ ]:
# FUNCTION: Plot legend for PR + ROC
def plot_pr_roc_legend(pr_roc_data):
    fig_legend, ax_leg = plt.subplots(figsize=(3, 3))
    ax_leg.axis('off')

    legend_lines = []
    legend_labels = []

    for method in pr_roc_data:
        legend_lines.append(plt.Line2D([0], [0], color=method_colors[method], lw=2))
        legend_labels.append(f"{method}\nPR AUC={pr_roc_data[method]['pr_auc']:.3f}")

    for sym, p in zip(threshold_symbols, threshold_points):
        legend_lines.append(plt.Line2D([0], [0], color='black', marker=sym, linestyle='None', markersize=8))
        legend_labels.append(f'p = {p:.3g}')

    ax_leg.legend(legend_lines, legend_labels, loc='center', frameon=False)
    plt.savefig('figleg.svg', format='svg')
    plt.show()

In [ ]:
### Figure 3A
# 1. Positive Associations
pr_roc_data_pos = collect_pr_roc_data(label_of_interest=1)
plot_pr_curves(pr_roc_data_pos, label_text='Positive')
plot_pr_roc_legend(pr_roc_data_pos)

In [ ]:
### Figure 3A
# 2. Negative Associations
pr_roc_data_neg = collect_pr_roc_data(label_of_interest=-1)
plot_pr_curves(pr_roc_data_neg, label_text='Negative')
plot_pr_roc_legend(pr_roc_data_neg)

In [ ]:
plot_pr_curves_zoom(pr_roc_data_pos, 'Positive')
plot_pr_curves_zoom(pr_roc_data_neg, 'Negative')